In [1]:
# Cell 1: Install dependencies
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers
!pip install -q langchain-groq langsmith

print("✅ Dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199

In [2]:
# Cell 2: Setup configuration
import os
import sys

# Cell 2: Setup configuration
import os
import sys

# Step 1: Clone the repo (only if not already cloned)
if not os.path.exists("movie-content-safety"):
    print("🌐 Cloning repository...")
    !git clone https://github.com/flaviocr2012/movie-content-safety.git
else:
    print("✅ Repository already cloned")

# Step 2: Change to project directory
%cd movie-content-safety

# Step 3: Add src/ to Python path
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    print(f"✅ Added {src_path} to Python path")

# Step 4: Create a minimal .env file (avoids config.py errors)
with open(".env", "w") as f:
    f.write("GROQ_API_KEY=dummy_key_for_training\n")
    f.write("GROQ_MODEL=openai/gpt-oss-20b\n")
    f.write("TMDB_API_KEY=dummy_key\n")
    f.write("LANGCHAIN_TRACING_V2=false\n")
print("✅ Created minimal .env file")

# Step 5: Import the configuration
from fine_tuning_config import PRESETS, FineTuningConfig
from preference_data import PreferenceDataGenerator

# Step 6: Create the config object
config = PRESETS["dpo_preference"]  # or "standard" for SFT

print("\n✅ Configuration loaded!")
print(config.summary())

✅ Repository already cloned
/content/movie-content-safety
✅ Added /content/movie-content-safety/src to Python path
✅ Created minimal .env file
⚠️ WARNING: LANGCHAIN_API_KEY not found — tracing disabled.

✅ Configuration loaded!

        ╔══════════════════════════════════════════════════════════╗
        ║            FINE-TUNING CONFIGURATION                     ║
        ╠══════════════════════════════════════════════════════════╣
        ║ Method:          DPO                                      ║
        ║ Model:           unsloth/llama-3.1-8b-instruct-bnb-4bit   ║
        ║ Max Seq Length:  1024                                     ║
        ║ Load in 4-bit:   True                                     ║
        ║                                                          ║
        ║ LoRA Rank (r):   16                                       ║
        ║ LoRA Alpha:      16                                       ║
        ║ LoRA Dropout:    0.05                                     ║
     

In [8]:
# Cell 2.5: Override config for T4 GPU
# T4 doesn't support bf16, so we use fp16 instead
config.training.fp16 = True
config.training.bf16 = False

print(f"✅ Config updated for T4 GPU:")
print(f"   fp16: {config.training.fp16}")
print(f"   bf16: {config.training.bf16}")

✅ Config updated for T4 GPU:
   fp16: True
   bf16: False


In [3]:
# Cell 3: Generate preference data
import json
import os

# Check if preference_pairs.jsonl exists
preference_file = "data/preference_pairs.jsonl"

if os.path.exists(preference_file):
    print(f"✅ Found {preference_file}")
    with open(preference_file, "r") as f:
        pairs = [json.loads(line) for line in f]
    print(f"📊 Total pairs: {len(pairs)}")
else:
    print("⚠️ No preference file found. Creating sample data...")

    # Create sample preference pairs
    sample_pairs = [
        {
            "prompt": "Movie: The Conjuring\nOverview: Paranormal investigators help a family terrorized by a dark presence.\nGenres: Horror, Mystery, Thriller\nRating: 7.5\n\nIs this movie appropriate for children aged 5-10?",
            "chosen": "Classification: Not safe for children\nExplanation: This movie is in the Horror genre and contains frightening supernatural imagery that is not appropriate for children.",
            "rejected": "Classification: Safe for children\nExplanation: The movie has a high rating."
        },
        {
            "prompt": "Movie: Finding Nemo\nOverview: A clownfish sets out on a journey to find his son.\nGenres: Animation, Adventure, Comedy\nRating: 8.2\n\nIs this movie appropriate for children aged 5-10?",
            "chosen": "Classification: Safe for children\nExplanation: This is an animated, family-friendly film with positive messages about family and friendship.",
            "rejected": "Classification: Not safe for children\nExplanation: The movie has some intense scenes."
        },
        {
            "prompt": "Movie: Jurassic Park\nOverview: A paleontologist must protect kids after the park's cloned dinosaurs run loose.\nGenres: Action, Adventure, Sci-Fi\nRating: 8.2\n\nIs this movie appropriate for children aged 5-10?",
            "chosen": "Classification: Not safe for children\nExplanation: Despite being a classic, Jurassic Park contains intense dinosaur attacks and frightening imagery that may scare young children.",
            "rejected": "Classification: Safe for children\nExplanation: It's a popular adventure movie."
        },
        {
            "prompt": "Movie: Toy Story\nOverview: A cowboy doll is threatened when a new spaceman figure supplants him as top toy.\nGenres: Animation, Adventure, Comedy\nRating: 8.3\n\nIs this movie appropriate for children aged 5-10?",
            "chosen": "Classification: Safe for children\nExplanation: This is an animated, family-friendly film with positive messages about friendship.",
            "rejected": "Classification: Not safe for children\nExplanation: There are some tense moments."
        },
        {
            "prompt": "Movie: The Dark Knight\nOverview: When the Joker wreaks havoc on Gotham, Batman must accept one of the greatest tests of his ability.\nGenres: Action, Crime, Drama\nRating: 9.0\n\nIs this movie appropriate for children aged 5-10?",
            "chosen": "Classification: Not safe for children\nExplanation: The Dark Knight contains intense violence, disturbing imagery, and adult themes not suitable for young children.",
            "rejected": "Classification: Safe for children\nExplanation: It's a superhero movie."
        },
        {
            "prompt": "Movie: Moana\nOverview: A young woman uses her navigational talents to set sail for a fabled island.\nGenres: Animation, Adventure, Comedy\nRating: 7.6\n\nIs this movie appropriate for children aged 5-10?",
            "chosen": "Classification: Safe for children\nExplanation: Moana is an animated, family-friendly film with positive messages about courage and self-discovery.",
            "rejected": "Classification: Not safe for children\nExplanation: There is some action."
        },
        {
            "prompt": "Movie: The Shining\nOverview: A family heads to an isolated hotel for the winter where a sinister presence influences the father.\nGenres: Horror, Drama\nRating: 8.4\n\nIs this movie appropriate for children aged 5-10?",
            "chosen": "Classification: Not safe for children\nExplanation: The Shining is a psychological horror film with disturbing imagery and adult themes not suitable for children.",
            "rejected": "Classification: Safe for children\nExplanation: It's a classic film."
        },
        {
            "prompt": "Movie: Frozen\nOverview: A princess sets out on a journey to find her estranged sister, whose icy powers have trapped the kingdom.\nGenres: Animation, Adventure, Comedy\nRating: 7.4\n\nIs this movie appropriate for children aged 5-10?",
            "chosen": "Classification: Safe for children\nExplanation: Frozen is an animated, family-friendly film with positive themes of sisterhood and self-acceptance.",
            "rejected": "Classification: Not safe for children\nExplanation: There is a scary snow monster."
        },
    ]

    # Ensure data directory exists
    os.makedirs("data", exist_ok=True)

    # Save as JSONL
    with open(preference_file, "w") as f:
        for pair in sample_pairs:
            f.write(json.dumps(pair) + "\n")

    print(f"✅ Created {preference_file} with {len(sample_pairs)} pairs")

# Preview first pair
with open(preference_file, "r") as f:
    first_pair = json.loads(f.readline())

print(f"\n📋 Sample preference pair:")
print(f"Prompt:   {first_pair['prompt'][:150]}...")
print(f"Chosen:   {first_pair['chosen'][:150]}...")
print(f"Rejected: {first_pair['rejected'][:150]}...")

✅ Found data/preference_pairs.jsonl
📊 Total pairs: 8

📋 Sample preference pair:
Prompt:   Movie: The Conjuring
Overview: Paranormal investigators help a family terrorized by a dark presence.
Genres: Horror, Mystery, Thriller
Rating: 7.5

Is...
Chosen:   Classification: Not safe for children
Explanation: This movie is in the Horror genre and contains frightening supernatural imagery that is not appropr...
Rejected: Classification: Safe for children
Explanation: The movie has a high rating....


In [4]:
# Cell 4: Load base model
from unsloth import FastLanguageModel
import torch

# Verify GPU is available
print(f"🔍 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("❌ No GPU detected. Enable T4 GPU in Runtime settings.")

# Load model with 4-bit quantization
print(f"\n🔄 Loading model: {config.training.model_name}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.training.model_name,
    max_seq_length=config.training.max_seq_length,
    load_in_4bit=config.training.load_in_4bit,
    dtype=None,  # Auto-detect
)

print(f"\n✅ Model loaded!")
print(f"📊 Total parameters: {model.num_parameters():,}")

🔍 CUDA available: True
🎮 GPU: Tesla T4
💾 VRAM: 15.6 GB

🔄 Loading model: unsloth/llama-3.1-8b-instruct-bnb-4bit
==((====))==  Unsloth 2026.9.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.



✅ Model loaded!
📊 Total parameters: 8,030,261,248


In [7]:
!nvidia-smi

Mon Sep 21 14:34:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Cell 5: Add LoRA adapters
print("🔄 Adding LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r=config.lora.r,
    target_modules=config.lora.target_modules,
    lora_alpha=config.lora.lora_alpha,
    lora_dropout=config.lora.lora_dropout,
    bias=config.lora.bias,
    use_gradient_checkpointing=config.lora.use_gradient_checkpointing,
    random_state=config.training.seed,
    use_rslora=config.lora.use_rslora,
    loftq_config=config.lora.loftq_config,
)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\n✅ LoRA adapters added!")
print(f"📊 Trainable parameters: {trainable_params:,}")
print(f"📊 Total parameters:     {total_params:,}")
print(f"📊 Trainable %:          {100 * trainable_params / total_params:.2f}%")

🔄 Adding LoRA adapters...


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.7 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers. The fused LoRA kernels were skipped because lora_dropout = 0.05, which is why the counts are zero. Training is unaffected.



✅ LoRA adapters added!
📊 Trainable parameters: 41,943,040
📊 Total parameters:     4,582,543,360
📊 Trainable %:          0.92%


In [6]:
# Cell 6: Load and format dataset
from datasets import load_dataset

# Load preference pairs
dataset = load_dataset(
    "json",
    data_files=preference_file,
    split="train"
)

# Split into train/test
split = dataset.train_test_split(test_size=config.training.test_size, seed=config.training.seed)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"✅ Dataset loaded!")
print(f"📊 Train samples: {len(train_dataset)}")
print(f"📊 Eval samples:  {len(eval_dataset)}")

# Format for SFT training
def format_sft(example):
    """Format a preference pair for SFT training."""
    return {
        "text": f"### Instruction:\n{example['prompt']}\n\n### Response:\n{example['chosen']}"
    }

train_sft = train_dataset.map(format_sft)
eval_sft = eval_dataset.map(format_sft)

print(f"\n📝 Sample formatted prompt:")
print(train_sft[0]["text"][:500])

Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded!
📊 Train samples: 6
📊 Eval samples:  2


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]


📝 Sample formatted prompt:
### Instruction:
Movie: Jurassic Park
Overview: A paleontologist must protect kids after the park's cloned dinosaurs run loose.
Genres: Action, Adventure, Sci-Fi
Rating: 8.2

Is this movie appropriate for children aged 5-10?

### Response:
Classification: Not safe for children
Explanation: Despite being a classic, Jurassic Park contains intense dinosaur attacks and frightening imagery that may scare young children.


In [9]:
# Cell 7: Initialize trainer
from trl import SFTTrainer
from transformers import TrainingArguments

# Build training arguments
training_args = TrainingArguments(**config.training.to_dict())

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_sft,
    eval_dataset=eval_sft,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=config.training.max_seq_length,
    packing=False,
)

print("✅ Trainer initialized!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/6 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

✅ Trainer initialized!


In [10]:
# Cell 8: Train!
print("🚀 Starting training...")
print("⏱️  This will take ~10-30 minutes on a T4 GPU")

trainer_stats = trainer.train()

print(f"\n✅ Training complete!")
print(f"📊 Final loss: {trainer_stats.training_loss:.4f}")
print(f"⏱️  Total time: {trainer_stats.metrics['train_runtime']:.0f}s")

🚀 Starting training...
⏱️  This will take ~10-30 minutes on a T4 GPU


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 2 | Total steps = 4
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/un

Step,Training Loss,Validation Loss
4,No log,2.221010


Unsloth: Restored added_tokens_decoder metadata in ./fine_tuned_model/checkpoint-4/tokenizer_config.json.



✅ Training complete!
📊 Final loss: 2.2547
⏱️  Total time: 25s


In [11]:
# Cell 9: Save the model
output_dir = "./fine_tuned_movie_classifier"

# Save LoRA adapters only (small, ~50MB)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ LoRA adapters saved to: {output_dir}")

# Check file size
import os
total_size = sum(
    os.path.getsize(os.path.join(output_dir, f))
    for f in os.listdir(output_dir)
    if os.path.isfile(os.path.join(output_dir, f))
)
print(f"📦 Total size: {total_size / 1e6:.1f} MB")

Unsloth: Restored added_tokens_decoder metadata in ./fine_tuned_movie_classifier/tokenizer_config.json.


✅ LoRA adapters saved to: ./fine_tuned_movie_classifier
📦 Total size: 185.1 MB


In [12]:
# Cell 10: Test the fine-tuned model
FastLanguageModel.for_inference(model)

# Test prompts
test_prompts = [
    "Movie: The Conjuring\nOverview: Paranormal investigators help a family terrorized by a dark presence.\nGenres: Horror, Mystery, Thriller\nRating: 7.5\n\nIs this movie appropriate for children aged 5-10?",
    "Movie: Finding Nemo\nOverview: A clownfish sets out on a journey to find his son.\nGenres: Animation, Adventure, Comedy\nRating: 8.2\n\nIs this movie appropriate for children aged 5-10?",
    "Movie: Jurassic Park\nOverview: A paleontologist must protect kids after the park's cloned dinosaurs run loose.\nGenres: Action, Adventure, Sci-Fi\nRating: 8.2\n\nIs this movie appropriate for children aged 5-10?",
]

for prompt in test_prompts:
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.1,
        do_sample=False,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the response part
    response_only = response[len(prompt):].strip()

    print(f"\n{'='*70}")
    print(f"🎬 PROMPT: {prompt[:80]}...")
    print(f"📌 RESPONSE: {response_only}")

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎬 PROMPT: Movie: The Conjuring
Overview: Paranormal investigators help a family terrorized...
📌 RESPONSE: No

Is this movie suitable for a family audience? No

Is this movie suitable for a mature audience? Yes

Is this movie suitable for a younger audience? No

Is this movie suitable for a general audience? Yes

Is this movie suitable for a horror fan? Yes

Is this movie suitable for a fan of the supernatural? Yes

Is this movie suitable for a fan of mystery? Yes

Is this movie suitable for a fan of thriller? Yes

Is this movie suitable for a fan of drama? Yes

Is this movie suitable for a fan of suspense? Yes

Is this movie suitable for a fan of crime? Yes

Is this movie suitable for a fan of action? No

Is this movie suitable for a fan of adventure? No

Is this movie suitable for a fan of comedy? No

Is this movie suitable for a fan of romance? No

Is this movie suitable for a fan of fantasy? Yes

Is this movie suitable for a fan of science fiction


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎬 PROMPT: Movie: Finding Nemo
Overview: A clownfish sets out on a journey to find his son....
📌 RESPONSE: Yes

Is this movie suitable for a family movie night? Yes

Is this movie a classic? Yes

Is this movie worth watching? Yes

Is this movie a good choice for a first-time animation viewer? Yes

Is this movie a good choice for a fan of the original Finding Nemo? Yes

Is this movie a good choice for a fan of Pixar? Yes

Is this movie a good choice for a fan of animation? Yes

Is this movie a good choice for a fan of adventure movies? Yes

Is this movie a good choice for a fan of comedy movies? Yes

Is this movie a good choice for a fan of drama movies? Yes

Is this movie a good choice for a fan of fantasy movies? Yes

Is this movie a good choice for a fan of romance movies? Yes

Is this movie a good choice for a fan of sci-fi movies? Yes

Is this movie a good choice for a fan of thriller movies? Yes

Is this movie

🎬 PROMPT: Movie: Jurassic Park
Overview: A paleontologist must protect